In [1]:
import pandas as pd
from oggm import utils

# Load geodetic data
utils.get_geodetic_mb_dataframe()
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

# Load your RGI cluster data
rgi_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/files_chile_OGGM_climate_comparison/RGI_BNA_Clusters.csv'
rgi_df = pd.read_csv(rgi_file)

# For each cluster, identify glaciers with/without GMB
cluster_gmb_status = []

for cluster in ['OT3', 'DA1', 'DA2', 'DA3', 'WA1', 'WA2', 'WA3', 'WA4', 'WA5', 'WA6']:
    # Get all glaciers in this cluster
    cluster_glaciers = rgi_df[rgi_df['Cluster'] == cluster]['RGIId'].tolist()
    
    # Check which have GMB
    has_gmb = [g for g in cluster_glaciers if g in geodetic_ref.index]
    no_gmb = [g for g in cluster_glaciers if g not in geodetic_ref.index]
    
    cluster_gmb_status.append({
        'cluster': cluster,
        'total_glaciers': len(cluster_glaciers),
        'with_gmb': len(has_gmb),
        'without_gmb': len(no_gmb),
        'pct_with_gmb': (len(has_gmb) / len(cluster_glaciers)) * 100
    })

status_df = pd.DataFrame(cluster_gmb_status)

print("\n" + "="*70)
print("GMB DATA AVAILABILITY BY CLUSTER")
print("="*70)
print(status_df.to_string(index=False))
print("="*70)

# Save lists for later use
glacier_lists = {}
for cluster in ['OT3', 'DA1', 'DA2', 'DA3', 'WA1', 'WA2', 'WA3', 'WA4', 'WA5', 'WA6']:
    cluster_glaciers = rgi_df[rgi_df['Cluster'] == cluster]['RGIId'].tolist()
    glacier_lists[cluster] = {
        'all': cluster_glaciers,
        'with_gmb': [g for g in cluster_glaciers if g in geodetic_ref.index],
        'without_gmb': [g for g in cluster_glaciers if g not in geodetic_ref.index]
    }

print(f"\n✓ Glacier lists created for all clusters")

/Users/milliespencer/miniconda3/envs/oggm_cr2_env/lib/python3.11/site-packages/oggm/__init__.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound



GMB DATA AVAILABILITY BY CLUSTER
cluster  total_glaciers  with_gmb  without_gmb  pct_with_gmb
    OT3              34        34            0         100.0
    DA1             422       422            0         100.0
    DA2             200       200            0         100.0
    DA3             410       410            0         100.0
    WA1             678       678            0         100.0
    WA2            2924      2924            0         100.0
    WA3             447       447            0         100.0
    WA4            4749      4749            0         100.0
    WA5             711       711            0         100.0
    WA6            2671      2671            0         100.0

✓ Glacier lists created for all clusters


In [2]:
import xarray as xr
import numpy as np

def calculate_smb_gmb_for_cluster(cluster, dataset='CR2MET'):
    """
    Calculate SMB and GMB using ONLY glaciers that have GMB data
    This ensures apples-to-apples comparison
    """
    # Load NetCDF
    if dataset == 'CR2MET':
        suffix = 'TC'
    else:
        suffix = dataset
    
    nc_file = f'/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/{dataset}/{cluster}/run_output_2000_2019_hydro_{suffix}_{cluster}.nc'
    ds = xr.open_dataset(nc_file)
    
    # Get glaciers with GMB data for this cluster
    glaciers_with_gmb = glacier_lists[cluster]['with_gmb']
    
    # Calculate SMB using ONLY these glaciers
    vol_2000 = 0
    vol_2020 = 0
    area_2000 = 0
    
    glaciers_in_nc = [str(x) for x in ds['rgi_id'].values]
    matched_glaciers = [g for g in glaciers_with_gmb if g in glaciers_in_nc]
    
    for rgi_id in matched_glaciers:
        vol_2000 += float(ds['volume'].sel(time=2000, rgi_id=rgi_id).values)
        vol_2020 += float(ds['volume'].sel(time=2020, rgi_id=rgi_id).values)
        area_2000 += float(ds['area'].sel(time=2000, rgi_id=rgi_id).values)
    
    # SMB in mm/yr
    smb = ((vol_2020 - vol_2000) / area_2000 / 20) * 1000
    
    # Calculate GMB for same glaciers (area-weighted)
    gmb = np.average(
        geodetic_ref.loc[matched_glaciers, 'dmdtda'] * 1000,
        weights=geodetic_ref.loc[matched_glaciers, 'area']
    )
    
    gmb_error = np.average(
        geodetic_ref.loc[matched_glaciers, 'err_dmdtda'] * 1000,
        weights=geodetic_ref.loc[matched_glaciers, 'area']
    )
    
    ds.close()
    
    return {
        'cluster': cluster,
        'dataset': dataset,
        'n_glaciers_total': len(glaciers_in_nc),
        'n_glaciers_with_gmb': len(matched_glaciers),
        'n_glaciers_compared': len(matched_glaciers),
        'SMB': smb,
        'GMB': gmb,
        'GMB_error': gmb_error,
        'bias': smb - gmb
    }

# Test for DA1 with all three datasets
print("\n" + "="*70)
print("CORRECTED COMPARISON - SAME GLACIER SETS")
print("="*70)

for dataset in ['CR2MET', 'CRU', 'ERA5']:
    result = calculate_smb_gmb_for_cluster('DA1', dataset)
    print(f"\n{dataset}:")
    print(f"  Total glaciers in simulation: {result['n_glaciers_total']}")
    print(f"  Glaciers compared (with GMB):  {result['n_glaciers_compared']}")
    print(f"  SMB: {result['SMB']:.1f} mm/yr")
    print(f"  GMB: {result['GMB']:.1f} ± {result['GMB_error']:.1f} mm/yr")
    print(f"  Bias: {result['bias']:+.1f} mm/yr")

print("\n" + "="*70)
print("Ale expects bias ≈ 0 mm/yr for all datasets!")
print("="*70)


CORRECTED COMPARISON - SAME GLACIER SETS

CR2MET:
  Total glaciers in simulation: 422
  Glaciers compared (with GMB):  422
  SMB: -93.8 mm/yr
  GMB: -98.7 ± 152.6 mm/yr
  Bias: +4.9 mm/yr

CRU:
  Total glaciers in simulation: 422
  Glaciers compared (with GMB):  422
  SMB: nan mm/yr
  GMB: -98.7 ± 152.6 mm/yr
  Bias: +nan mm/yr

ERA5:
  Total glaciers in simulation: 422
  Glaciers compared (with GMB):  422
  SMB: nan mm/yr
  GMB: -98.7 ± 152.6 mm/yr
  Bias: +nan mm/yr

Ale expects bias ≈ 0 mm/yr for all datasets!


In [3]:
import xarray as xr
import pandas as pd
import numpy as np
from oggm import utils

# Load NetCDF for DA1
nc_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc'
ds = xr.open_dataset(nc_file)

# Load geodetic data
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

# Load RGI cluster data
rgi_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/files_chile_OGGM_climate_comparison/RGI_BNA_Clusters.csv'
rgi_df = pd.read_csv(rgi_file)

print("\n" + "="*70)
print("DETECTIVE WORK - WHERE'S THE MISMATCH?")
print("="*70)

# 1. What's in the NetCDF?
nc_glaciers = [str(x) for x in ds['rgi_id'].values]
print(f"\n1. NetCDF file contains: {len(nc_glaciers)} glaciers")

# 2. What's in the RGI cluster file?
rgi_da1 = rgi_df[rgi_df['Cluster'] == 'DA1']['RGIId'].tolist()
print(f"2. RGI cluster file says DA1 has: {len(rgi_da1)} glaciers")

# 3. What's in GMB dataset?
gmb_da1 = [g for g in rgi_da1 if g in geodetic_ref.index]
print(f"3. GMB dataset has data for: {len(gmb_da1)} glaciers")

# 4. Check overlaps
nc_in_rgi = [g for g in nc_glaciers if g in rgi_da1]
nc_in_gmb = [g for g in nc_glaciers if g in geodetic_ref.index]
nc_in_both = [g for g in nc_glaciers if g in rgi_da1 and g in geodetic_ref.index]

print(f"\n4. Glacier set overlaps:")
print(f"   NetCDF glaciers also in RGI file: {len(nc_in_rgi)}")
print(f"   NetCDF glaciers also in GMB data: {len(nc_in_gmb)}")
print(f"   NetCDF glaciers in BOTH RGI and GMB: {len(nc_in_both)}")

# 5. Are they the same glaciers?
print(f"\n5. Set comparisons:")
print(f"   NetCDF == RGI? {set(nc_glaciers) == set(rgi_da1)}")
print(f"   NetCDF == GMB? {set(nc_glaciers) == set(gmb_da1)}")
print(f"   RGI == GMB? {set(rgi_da1) == set(gmb_da1)}")

# 6. What's different?
nc_not_in_gmb = [g for g in nc_glaciers if g not in geodetic_ref.index]
gmb_not_in_nc = [g for g in gmb_da1 if g not in nc_glaciers]

print(f"\n6. What's missing where?")
print(f"   Glaciers in NetCDF but NOT in GMB: {len(nc_not_in_gmb)}")
if len(nc_not_in_gmb) > 0:
    print(f"      Examples: {nc_not_in_gmb[:5]}")
    
print(f"   Glaciers in GMB but NOT in NetCDF: {len(gmb_not_in_nc)}")
if len(gmb_not_in_nc) > 0:
    print(f"      Examples: {gmb_not_in_nc[:5]}")

# 7. Calculate SMB and GMB using different methods
print(f"\n" + "="*70)
print("COMPARING CALCULATION METHODS")
print("="*70)

# Method A: ALL NetCDF glaciers (your original cluster method)
vol_2000_all = ds['volume'].sel(time=2000).sum()
vol_2020_all = ds['volume'].sel(time=2020).sum()
area_2000_all = ds['area'].sel(time=2000).sum()
smb_all = float((vol_2020_all - vol_2000_all) / area_2000_all / 20 * 1000)

print(f"\nMethod A - ALL NetCDF glaciers ({len(nc_glaciers)}):")
print(f"  SMB: {smb_all:.1f} mm/yr")

# Method B: Only NetCDF glaciers that are ALSO in GMB
vol_2000_matched = 0
vol_2020_matched = 0
area_2000_matched = 0

for rgi_id in nc_in_gmb:
    vol_2000_matched += float(ds['volume'].sel(time=2000, rgi_id=rgi_id).values)
    vol_2020_matched += float(ds['volume'].sel(time=2020, rgi_id=rgi_id).values)
    area_2000_matched += float(ds['area'].sel(time=2000, rgi_id=rgi_id).values)

smb_matched = ((vol_2020_matched - vol_2000_matched) / area_2000_matched / 20) * 1000

print(f"\nMethod B - Only NetCDF glaciers in GMB ({len(nc_in_gmb)}):")
print(f"  SMB: {smb_matched:.1f} mm/yr")

# GMB from matched glaciers
if len(nc_in_gmb) > 0:
    gmb_matched = np.average(
        geodetic_ref.loc[nc_in_gmb, 'dmdtda'] * 1000,
        weights=geodetic_ref.loc[nc_in_gmb, 'area']
    )
    print(f"  GMB: {gmb_matched:.1f} mm/yr")
    print(f"  Bias: {smb_matched - gmb_matched:+.1f} mm/yr")

# GMB from ALL DA1 glaciers (even if not in NetCDF)
gmb_all = np.average(
    geodetic_ref.loc[gmb_da1, 'dmdtda'] * 1000,
    weights=geodetic_ref.loc[gmb_da1, 'area']
)

print(f"\nMethod C - GMB for ALL DA1 RGI glaciers ({len(gmb_da1)}):")
print(f"  GMB: {gmb_all:.1f} mm/yr")
print(f"  Bias (vs SMB all): {smb_all - gmb_all:+.1f} mm/yr")

print("\n" + "="*70)

ds.close()


DETECTIVE WORK - WHERE'S THE MISMATCH?

1. NetCDF file contains: 422 glaciers
2. RGI cluster file says DA1 has: 422 glaciers
3. GMB dataset has data for: 422 glaciers

4. Glacier set overlaps:
   NetCDF glaciers also in RGI file: 422
   NetCDF glaciers also in GMB data: 422
   NetCDF glaciers in BOTH RGI and GMB: 422

5. Set comparisons:
   NetCDF == RGI? True
   NetCDF == GMB? True
   RGI == GMB? True

6. What's missing where?
   Glaciers in NetCDF but NOT in GMB: 0
   Glaciers in GMB but NOT in NetCDF: 0

COMPARING CALCULATION METHODS

Method A - ALL NetCDF glaciers (422):
  SMB: -93.8 mm/yr

Method B - Only NetCDF glaciers in GMB (422):
  SMB: -93.8 mm/yr
  GMB: -98.7 mm/yr
  Bias: +4.9 mm/yr

Method C - GMB for ALL DA1 RGI glaciers (422):
  GMB: -98.7 mm/yr
  Bias (vs SMB all): +4.9 mm/yr



In [5]:
def calculate_smb_gmb_for_cluster_DEBUG(cluster, dataset='CR2MET'):
    """
    Calculate SMB and GMB with detailed debugging
    """
    base_path = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/'
    
    if dataset == 'CR2MET':
        nc_file = f'{base_path}CR2MET/{cluster}/run_output_2000_2019_hydro_TC_{cluster}.nc'
    elif dataset == 'CRU':
        nc_file = f'{base_path}CRU/{cluster}/run_output_2000_2019_hydro_CRU_{cluster}.nc'
    elif dataset == 'ERA5':
        nc_file = f'{base_path}ERA5/{cluster}/run_output_2000_2019_hydro_ERA5_{cluster}.nc'
    
    ds = xr.open_dataset(nc_file)
    
    # Check what's in the NetCDF
    print(f"\n{dataset} NetCDF contents:")
    print(f"  Variables: {list(ds.data_vars)}")
    print(f"  Dimensions: {dict(ds.dims)}")
    print(f"  Time values: {ds['time'].values}")
    
    # Check if 2000 and 2020 are in time
    has_2000 = 2000 in ds['time'].values
    has_2020 = 2020 in ds['time'].values
    print(f"  Has year 2000: {has_2000}")
    print(f"  Has year 2020: {has_2020}")
    
    if not has_2000 or not has_2020:
        print(f"  ✗ Missing required years!")
        ds.close()
        return None
    
    # Try to extract volumes
    try:
        vol_2000 = ds['volume'].sel(time=2000).sum()
        vol_2020 = ds['volume'].sel(time=2020).sum()
        area_2000 = ds['area'].sel(time=2000).sum()
        
        print(f"  Volume 2000: {float(vol_2000):,.0f} m³")
        print(f"  Volume 2020: {float(vol_2020):,.0f} m³")
        print(f"  Area 2000: {float(area_2000):,.0f} m²")
        
        smb = float((vol_2020 - vol_2000) / area_2000 / 20 * 1000)
        print(f"  Calculated SMB: {smb:.1f} mm/yr")
        
    except Exception as e:
        print(f"  ✗ Error calculating SMB: {e}")
        import traceback
        traceback.print_exc()
        ds.close()
        return None
    
    ds.close()
    return smb

# Test each dataset
print("\n" + "="*70)
print("DETAILED DEBUGGING")
print("="*70)

for dataset in ['CR2MET', 'CRU', 'ERA5']:
    smb = calculate_smb_gmb_for_cluster_DEBUG('DA1', dataset)
    if smb is not None:
        print(f"  → Final SMB: {smb:.1f} mm/yr\n")


DETAILED DEBUGGING

CR2MET NetCDF contents:
  Variables: ['volume', 'volume_bsl', 'volume_bwl', 'area', 'length', 'calving', 'calving_rate', 'off_area', 'on_area', 'melt_off_glacier', 'melt_on_glacier', 'liq_prcp_off_glacier', 'liq_prcp_on_glacier', 'snowfall_off_glacier', 'snowfall_on_glacier', 'melt_off_glacier_monthly', 'melt_on_glacier_monthly', 'liq_prcp_off_glacier_monthly', 'liq_prcp_on_glacier_monthly', 'snowfall_off_glacier_monthly', 'snowfall_on_glacier_monthly', 'water_level', 'glen_a', 'fs']
  Dimensions: {'time': 24, 'rgi_id': 422, 'month_2d': 12}
  Time values: [1999. 2000. 2001. 2002. 2003. 2004. 2005. 2006. 2007. 2008. 2009. 2010.
 2011. 2012. 2013. 2014. 2015. 2016. 2017. 2018. 2019. 2020. 2021. 2022.]
  Has year 2000: True
  Has year 2020: True
  Volume 2000: 5,626,693,632 m³
  Volume 2020: 5,343,352,832 m³
  Area 2000: 151,072,896 m²
  Calculated SMB: -93.8 mm/yr
  → Final SMB: -93.8 mm/yr


CRU NetCDF contents:
  Variables: ['volume', 'volume_bsl', 'volume_bwl', 'a

/var/folders/3j/6dy_9gxj7vvgct178jkkp1680000gn/T/ipykernel_57930/3732999059.py:19: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds.dims)}")
/var/folders/3j/6dy_9gxj7vvgct178jkkp1680000gn/T/ipykernel_57930/3732999059.py:19: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds.dims)}")
/var/folders/3j/6dy_9gxj7vvgct178jkkp1680000gn/T/ipykernel_57930/3732999059.py:19: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access 

In [6]:
# Quick check: Are we using the same area for weighting?
import xarray as xr
import numpy as np

ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')
ds_cru = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CRU/DA1/run_output_2000_2019_hydro_CRU_DA1.nc')

# Compare initial areas
area_cr2 = float(ds_cr2['area'].sel(time=2000).sum())
area_cru = float(ds_cru['area'].sel(time=2000).sum())
area_gmb = geodetic_ref.loc[glacier_lists['DA1']['with_gmb'], 'area'].sum()

print(f"\nArea comparison:")
print(f"  CR2MET area (2000): {area_cr2/1e6:.2f} km²")
print(f"  CRU area (2000):    {area_cru/1e6:.2f} km²")
print(f"  GMB area:           {area_gmb:.2f} km²")

ds_cr2.close()
ds_cru.close()


Area comparison:
  CR2MET area (2000): 151.07 km²
  CRU area (2000):    148.89 km²
  GMB area:           148094000.00 km²


In [7]:
# CORRECTED area comparison
area_gmb_m2 = geodetic_ref.loc[glacier_lists['DA1']['with_gmb'], 'area'].sum()
area_gmb_km2 = area_gmb_m2 / 1e6  # Convert m² to km²

print(f"\nCORRECTED Area comparison:")
print(f"  CR2MET area (2000): {area_cr2/1e6:.2f} km²")
print(f"  CRU area (2000):    {area_cru/1e6:.2f} km²")
print(f"  GMB area:           {area_gmb_km2:.2f} km²")

# Now they should match!


CORRECTED Area comparison:
  CR2MET area (2000): 151.07 km²
  CRU area (2000):    148.89 km²
  GMB area:           148.09 km²


In [8]:
import xarray as xr
import pandas as pd
import numpy as np
from oggm import utils

# Load NetCDF for DA1
nc_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc'
ds = xr.open_dataset(nc_file)

# Load geodetic data
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

# Load RGI cluster data
rgi_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/files_chile_OGGM_climate_comparison/RGI_BNA_Clusters.csv'
rgi_df = pd.read_csv(rgi_file)

print("\n" + "="*70)
print("DETECTIVE WORK - WHERE'S THE MISMATCH?")
print("="*70)

# 1. What's in the NetCDF?
nc_glaciers = [str(x) for x in ds['rgi_id'].values]
print(f"\n1. NetCDF file contains: {len(nc_glaciers)} glaciers")

# 2. What's in the RGI cluster file?
rgi_da1 = rgi_df[rgi_df['Cluster'] == 'DA1']['RGIId'].tolist()
print(f"2. RGI cluster file says DA1 has: {len(rgi_da1)} glaciers")

# 3. What's in GMB dataset?
gmb_da1 = [g for g in rgi_da1 if g in geodetic_ref.index]
print(f"3. GMB dataset has data for: {len(gmb_da1)} glaciers")

# 4. Check overlaps
nc_in_rgi = [g for g in nc_glaciers if g in rgi_da1]
nc_in_gmb = [g for g in nc_glaciers if g in geodetic_ref.index]
nc_in_both = [g for g in nc_glaciers if g in rgi_da1 and g in geodetic_ref.index]

print(f"\n4. Glacier set overlaps:")
print(f"   NetCDF glaciers also in RGI file: {len(nc_in_rgi)}")
print(f"   NetCDF glaciers also in GMB data: {len(nc_in_gmb)}")
print(f"   NetCDF glaciers in BOTH RGI and GMB: {len(nc_in_both)}")

# 5. Are they the same glaciers?
print(f"\n5. Set comparisons:")
print(f"   NetCDF == RGI? {set(nc_glaciers) == set(rgi_da1)}")
print(f"   NetCDF == GMB? {set(nc_glaciers) == set(gmb_da1)}")
print(f"   RGI == GMB? {set(rgi_da1) == set(gmb_da1)}")

# 6. What's different?
nc_not_in_gmb = [g for g in nc_glaciers if g not in geodetic_ref.index]
gmb_not_in_nc = [g for g in gmb_da1 if g not in nc_glaciers]

print(f"\n6. What's missing where?")
print(f"   Glaciers in NetCDF but NOT in GMB: {len(nc_not_in_gmb)}")
if len(nc_not_in_gmb) > 0:
    print(f"      Examples: {nc_not_in_gmb[:5]}")
    
print(f"   Glaciers in GMB but NOT in NetCDF: {len(gmb_not_in_nc)}")
if len(gmb_not_in_nc) > 0:
    print(f"      Examples: {gmb_not_in_nc[:5]}")

# 7. Calculate SMB and GMB using different methods
print(f"\n" + "="*70)
print("COMPARING CALCULATION METHODS")
print("="*70)

# Method A: ALL NetCDF glaciers (your original cluster method)
vol_2000_all = ds['volume'].sel(time=2000).sum()
vol_2020_all = ds['volume'].sel(time=2020).sum()
area_2000_all = ds['area'].sel(time=2000).sum()
smb_all = float((vol_2020_all - vol_2000_all) / area_2000_all / 20 * 1000)

print(f"\nMethod A - ALL NetCDF glaciers ({len(nc_glaciers)}):")
print(f"  SMB: {smb_all:.1f} mm/yr")

# Method B: Only NetCDF glaciers that are ALSO in GMB
vol_2000_matched = 0
vol_2020_matched = 0
area_2000_matched = 0

for rgi_id in nc_in_gmb:
    vol_2000_matched += float(ds['volume'].sel(time=2000, rgi_id=rgi_id).values)
    vol_2020_matched += float(ds['volume'].sel(time=2020, rgi_id=rgi_id).values)
    area_2000_matched += float(ds['area'].sel(time=2000, rgi_id=rgi_id).values)

smb_matched = ((vol_2020_matched - vol_2000_matched) / area_2000_matched / 20) * 1000

print(f"\nMethod B - Only NetCDF glaciers in GMB ({len(nc_in_gmb)}):")
print(f"  SMB: {smb_matched:.1f} mm/yr")

# GMB from matched glaciers
if len(nc_in_gmb) > 0:
    gmb_matched = np.average(
        geodetic_ref.loc[nc_in_gmb, 'dmdtda'] * 1000,
        weights=geodetic_ref.loc[nc_in_gmb, 'area']
    )
    print(f"  GMB: {gmb_matched:.1f} mm/yr")
    print(f"  Bias: {smb_matched - gmb_matched:+.1f} mm/yr")

# GMB from ALL DA1 glaciers (even if not in NetCDF)
gmb_all = np.average(
    geodetic_ref.loc[gmb_da1, 'dmdtda'] * 1000,
    weights=geodetic_ref.loc[gmb_da1, 'area']
)

print(f"\nMethod C - GMB for ALL DA1 RGI glaciers ({len(gmb_da1)}):")
print(f"  GMB: {gmb_all:.1f} mm/yr")
print(f"  Bias (vs SMB all): {smb_all - gmb_all:+.1f} mm/yr")

print("\n" + "="*70)

ds.close()


DETECTIVE WORK - WHERE'S THE MISMATCH?

1. NetCDF file contains: 422 glaciers
2. RGI cluster file says DA1 has: 422 glaciers
3. GMB dataset has data for: 422 glaciers

4. Glacier set overlaps:
   NetCDF glaciers also in RGI file: 422
   NetCDF glaciers also in GMB data: 422
   NetCDF glaciers in BOTH RGI and GMB: 422

5. Set comparisons:
   NetCDF == RGI? True
   NetCDF == GMB? True
   RGI == GMB? True

6. What's missing where?
   Glaciers in NetCDF but NOT in GMB: 0
   Glaciers in GMB but NOT in NetCDF: 0

COMPARING CALCULATION METHODS

Method A - ALL NetCDF glaciers (422):
  SMB: -93.8 mm/yr

Method B - Only NetCDF glaciers in GMB (422):
  SMB: -93.8 mm/yr
  GMB: -98.7 mm/yr
  Bias: +4.9 mm/yr

Method C - GMB for ALL DA1 RGI glaciers (422):
  GMB: -98.7 mm/yr
  Bias (vs SMB all): +4.9 mm/yr



In [9]:
import xarray as xr
import pandas as pd

# Load all three NetCDF files
ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')
ds_cru = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CRU/DA1/run_output_2000_2019_hydro_CRU_DA1.nc')
ds_era5 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/ERA5/DA1/run_output_2000_2019_hydro_ERA5_DA1.nc')

print("\n" + "="*70)
print("INVESTIGATING INITIAL AREA DIFFERENCES")
print("="*70)

# Check time dimension - what's the ACTUAL initial year?
print(f"\nTime dimensions:")
print(f"  CR2MET years: {ds_cr2['time'].values}")
print(f"  CRU years:    {ds_cru['time'].values}")
print(f"  ERA5 years:   {ds_era5['time'].values}")

# Compare areas at FIRST timestep (should be identical - the inventory)
print(f"\n{'Year':<10} {'CR2MET (km²)':<15} {'CRU (km²)':<15} {'ERA5 (km²)':<15}")
print("-" * 55)

for year in [1999, 2000, 2001]:
    cr2_area = float(ds_cr2['area'].sel(time=year).sum()) / 1e6
    cru_area = float(ds_cru['area'].sel(time=year).sum()) / 1e6
    era5_area = float(ds_era5['area'].sel(time=year).sum()) / 1e6
    print(f"{year:<10} {cr2_area:<15.2f} {cru_area:<15.2f} {era5_area:<15.2f}")

# Check individual glacier comparison for first 5 glaciers
print(f"\n{'='*70}")
print("INDIVIDUAL GLACIER AREAS (Year 1999 - Initial State)")
print(f"{'='*70}")

glacier_ids = list(ds_cr2['rgi_id'].values[:5])
print(f"\n{'RGI ID':<20} {'CR2MET (km²)':<15} {'CRU (km²)':<15} {'ERA5 (km²)':<15}")
print("-" * 70)

for rgi_id in glacier_ids:
    cr2_area = float(ds_cr2['area'].sel(time=1999, rgi_id=rgi_id).values) / 1e6
    cru_area = float(ds_cru['area'].sel(time=1999, rgi_id=rgi_id).values) / 1e6
    era5_area = float(ds_era5['area'].sel(time=1999, rgi_id=rgi_id).values) / 1e6
    
    print(f"{str(rgi_id):<20} {cr2_area:<15.4f} {cru_area:<15.4f} {era5_area:<15.4f}")

# Check if they're identical
cr2_1999_total = float(ds_cr2['area'].sel(time=1999).sum())
cru_1999_total = float(ds_cru['area'].sel(time=1999).sum())
era5_1999_total = float(ds_era5['area'].sel(time=1999).sum())

print(f"\n{'='*70}")
if abs(cr2_1999_total - cru_1999_total) < 1 and abs(cr2_1999_total - era5_1999_total) < 1:
    print("✅ Year 1999 areas are IDENTICAL across all datasets")
    print("   → Datasets start with same inventory")
    print("   → Differences in 2000 are due to 1 year of simulation (1999→2000)")
else:
    print("❌ Year 1999 areas are DIFFERENT")
    print("   → Datasets may be using different inventories!")

print(f"\nDifferences:")
print(f"  CR2MET 1999: {cr2_1999_total/1e6:.2f} km²")
print(f"  CRU 1999:    {cru_1999_total/1e6:.2f} km²")
print(f"  ERA5 1999:   {era5_1999_total/1e6:.2f} km²")
print(f"  Diff (CR2-CRU):  {(cr2_1999_total - cru_1999_total)/1e6:.4f} km²")
print(f"  Diff (CR2-ERA5): {(cr2_1999_total - era5_1999_total)/1e6:.4f} km²")

ds_cr2.close()
ds_cru.close()
ds_era5.close()

print(f"{'='*70}\n")


INVESTIGATING INITIAL AREA DIFFERENCES

Time dimensions:
  CR2MET years: [1999. 2000. 2001. 2002. 2003. 2004. 2005. 2006. 2007. 2008. 2009. 2010.
 2011. 2012. 2013. 2014. 2015. 2016. 2017. 2018. 2019. 2020. 2021. 2022.]
  CRU years:    [1999. 2000. 2001. 2002. 2003. 2004. 2005. 2006. 2007. 2008. 2009. 2010.
 2011. 2012. 2013. 2014. 2015. 2016. 2017. 2018. 2019. 2020.]
  ERA5 years:   [1999. 2000. 2001. 2002. 2003. 2004. 2005. 2006. 2007. 2008. 2009. 2010.
 2011. 2012. 2013. 2014. 2015. 2016. 2017. 2018. 2019. 2020.]

Year       CR2MET (km²)    CRU (km²)       ERA5 (km²)     
-------------------------------------------------------
1999       147.14          147.19          147.07         
2000       151.07          148.89          148.47         
2001       154.21          156.32          148.88         

INDIVIDUAL GLACIER AREAS (Year 1999 - Initial State)

RGI ID               CR2MET (km²)    CRU (km²)       ERA5 (km²)     
------------------------------------------------------------

In [10]:
import xarray as xr
import pandas as pd
import numpy as np

# Load all three NetCDF files
ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')
ds_cru = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CRU/DA1/run_output_2000_2019_hydro_CRU_DA1.nc')
ds_era5 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/ERA5/DA1/run_output_2000_2019_hydro_ERA5_DA1.nc')

print("\n" + "="*70)
print("CHECKING IF YEAR 2000 AREAS ARE IDENTICAL PER GLACIER")
print("="*70)

# Get all glacier IDs
glacier_ids = [str(x) for x in ds_cr2['rgi_id'].values]

# Check each glacier's year 2000 area
differences = []

print(f"\nChecking all {len(glacier_ids)} glaciers...")

for rgi_id in glacier_ids:
    cr2_area = float(ds_cr2['area'].sel(time=2000, rgi_id=rgi_id).values)
    cru_area = float(ds_cru['area'].sel(time=2000, rgi_id=rgi_id).values)
    era5_area = float(ds_era5['area'].sel(time=2000, rgi_id=rgi_id).values)
    
    # Check if they differ
    max_diff = max(abs(cr2_area - cru_area), abs(cr2_area - era5_area), abs(cru_area - era5_area))
    
    if max_diff > 1:  # More than 1 m² difference
        differences.append({
            'rgi_id': rgi_id,
            'CR2MET': cr2_area / 1e6,  # Convert to km²
            'CRU': cru_area / 1e6,
            'ERA5': era5_area / 1e6,
            'max_diff_m2': max_diff,
            'max_diff_pct': (max_diff / cr2_area) * 100
        })

if len(differences) == 0:
    print("\n✅ ALL GLACIERS HAVE IDENTICAL YEAR 2000 AREAS!")
    print("   (within 1 m² tolerance)")
else:
    print(f"\n❌ Found {len(differences)} glaciers with different year 2000 areas:")
    print(f"\n{'RGI ID':<20} {'CR2MET (km²)':<15} {'CRU (km²)':<15} {'ERA5 (km²)':<15} {'Max Diff (m²)':<15} {'% Diff':<10}")
    print("-" * 100)
    
    diff_df = pd.DataFrame(differences)
    for _, row in diff_df.head(20).iterrows():  # Show first 20
        print(f"{row['rgi_id']:<20} {row['CR2MET']:<15.4f} {row['CRU']:<15.4f} {row['ERA5']:<15.4f} {row['max_diff_m2']:<15.1f} {row['max_diff_pct']:<10.2f}")
    
    if len(differences) > 20:
        print(f"... and {len(differences) - 20} more")
    
    # Summary statistics
    print(f"\n{'='*70}")
    print("SUMMARY OF DIFFERENCES:")
    print(f"  Glaciers with differences: {len(differences)} / {len(glacier_ids)} ({len(differences)/len(glacier_ids)*100:.1f}%)")
    print(f"  Mean difference: {diff_df['max_diff_m2'].mean():.1f} m²")
    print(f"  Max difference: {diff_df['max_diff_m2'].max():.1f} m²")
    print(f"  Mean % difference: {diff_df['max_diff_pct'].mean():.2f}%")

# Check total cluster area
cr2_total = float(ds_cr2['area'].sel(time=2000).sum())
cru_total = float(ds_cru['area'].sel(time=2000).sum())
era5_total = float(ds_era5['area'].sel(time=2000).sum())

print(f"\n{'='*70}")
print("TOTAL CLUSTER AREA (Year 2000):")
print(f"  CR2MET: {cr2_total/1e6:.2f} km²")
print(f"  CRU:    {cru_total/1e6:.2f} km²")
print(f"  ERA5:   {era5_total/1e6:.2f} km²")
print(f"  Difference (CR2-CRU):  {(cr2_total - cru_total)/1e6:.2f} km² ({((cr2_total - cru_total)/cr2_total)*100:.1f}%)")
print(f"  Difference (CR2-ERA5): {(cr2_total - era5_total)/1e6:.2f} km² ({((cr2_total - era5_total)/cr2_total)*100:.1f}%)")
print(f"{'='*70}\n")

ds_cr2.close()
ds_cru.close()
ds_era5.close()


CHECKING IF YEAR 2000 AREAS ARE IDENTICAL PER GLACIER

Checking all 422 glaciers...

❌ Found 421 glaciers with different year 2000 areas:

RGI ID               CR2MET (km²)    CRU (km²)       ERA5 (km²)      Max Diff (m²)   % Diff    
----------------------------------------------------------------------------------------------------
RGI60-17.14542       0.3640          0.3643          0.3643          319.4           0.09      
RGI60-17.14547       0.0314          0.0310          0.0310          418.9           1.33      
RGI60-17.14549       0.0443          0.0431          0.0424          1891.2          4.27      
RGI60-17.14559       0.0308          0.0308          0.0307          66.8            0.22      
RGI60-17.14563       0.0361          0.0363          0.0361          189.1           0.52      
RGI60-17.14602       0.7681          0.7754          0.7832          15098.5         1.97      
RGI60-17.14607       0.0390          0.0441          0.0367          7376.9          18

In [14]:
import xarray as xr
import pandas as pd
import numpy as np

# Load all three NetCDF files
ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')
ds_cru = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CRU/DA1/run_output_2000_2019_hydro_CRU_DA1.nc')
ds_era5 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/ERA5/DA1/run_output_2000_2019_hydro_ERA5_DA1.nc')

print("\n" + "="*70)
print("CHECKING IF ALL YEAR 1999 AREAS ARE IDENTICAL")
print("="*70)

# Get all glacier IDs
glacier_ids = [str(x) for x in ds_cr2['rgi_id'].values]

# Check each glacier's year 1999 area
differences = []
identical_count = 0

print(f"\nChecking all {len(glacier_ids)} glaciers at year 1999...")

for rgi_id in glacier_ids:
    cr2_area_1999 = float(ds_cr2['area'].sel(time=1999, rgi_id=rgi_id).values)
    cru_area_1999 = float(ds_cru['area'].sel(time=1999, rgi_id=rgi_id).values)
    era5_area_1999 = float(ds_era5['area'].sel(time=1999, rgi_id=rgi_id).values)
    
    # Check if they're identical (within 1 m² tolerance for floating point)
    max_diff = max(abs(cr2_area_1999 - cru_area_1999), 
                   abs(cr2_area_1999 - era5_area_1999), 
                   abs(cru_area_1999 - era5_area_1999))
    
    if max_diff < 1:  # Less than 1 m² difference = identical
        identical_count += 1
    else:
        differences.append({
            'rgi_id': rgi_id,
            'CR2MET': cr2_area_1999 / 1e6,  # km²
            'CRU': cru_area_1999 / 1e6,
            'ERA5': era5_area_1999 / 1e6,
            'max_diff_m2': max_diff
        })

print(f"\n{'='*70}")
if len(differences) == 0:
    print("✅ ✅ ✅ ALL 422 GLACIERS HAVE IDENTICAL YEAR 1999 AREAS! ✅ ✅ ✅")
    print("   → All datasets start from same RGI inventory")
    print("   → Year 1999 in NetCDF = RGI ~2000 inventory")
else:
    print(f"❌ Found {len(differences)} glaciers with different year 1999 areas:")
    print(f"\n{'RGI ID':<20} {'CR2MET (km²)':<15} {'CRU (km²)':<15} {'ERA5 (km²)':<15} {'Diff (m²)':<12}")
    print("-" * 80)
    
    diff_df = pd.DataFrame(differences)
    for _, row in diff_df.head(20).iterrows():
        print(f"{row['rgi_id']:<20} {row['CR2MET']:<15.4f} {row['CRU']:<15.4f} {row['ERA5']:<15.4f} {row['max_diff_m2']:<12.1f}")

print(f"\n{'='*70}")
print(f"SUMMARY:")
print(f"  Identical glaciers (year 1999): {identical_count} / {len(glacier_ids)} ({identical_count/len(glacier_ids)*100:.1f}%)")
print(f"  Different glaciers (year 1999):  {len(differences)} / {len(glacier_ids)}")

# Check totals
cr2_total_1999 = float(ds_cr2['area'].sel(time=1999).sum()) / 1e6
cru_total_1999 = float(ds_cru['area'].sel(time=1999).sum()) / 1e6
era5_total_1999 = float(ds_era5['area'].sel(time=1999).sum()) / 1e6

print(f"\nTotal cluster area (year 1999):")
print(f"  CR2MET: {cr2_total_1999:.2f} km²")
print(f"  CRU:    {cru_total_1999:.2f} km²")
print(f"  ERA5:   {era5_total_1999:.2f} km²")
print(f"  Max difference: {max(abs(cr2_total_1999-cru_total_1999), abs(cr2_total_1999-era5_total_1999)):.4f} km²")

print(f"{'='*70}\n")

ds_cr2.close()
ds_cru.close()
ds_era5.close()


CHECKING IF ALL YEAR 1999 AREAS ARE IDENTICAL

Checking all 422 glaciers at year 1999...

❌ Found 15 glaciers with different year 1999 areas:

RGI ID               CR2MET (km²)    CRU (km²)       ERA5 (km²)      Diff (m²)   
--------------------------------------------------------------------------------
RGI60-17.14602       0.7680          0.7750          0.7833          15357.8     
RGI60-17.14926       0.6214          0.6214          0.6315          10116.1     
RGI60-17.14939       0.5365          0.5530          0.5530          16511.4     
RGI60-17.14994       1.0410          1.0165          1.0410          24527.6     
RGI60-17.15038       3.1400          3.1400          3.1059          34080.5     
RGI60-17.15077       0.0320          0.0000          0.0320          32000.0     
RGI60-17.15112       0.0400          0.0000          0.0400          40000.0     
RGI60-17.15140       1.1537          1.0983          1.1739          75590.0     
RGI60-17.15452       0.0490          

In [15]:
import xarray as xr
import pandas as pd

ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')
ds_cru = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CRU/DA1/run_output_2000_2019_hydro_CRU_DA1.nc')
ds_era5 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/ERA5/DA1/run_output_2000_2019_hydro_ERA5_DA1.nc')

# Problem glaciers
problem_ids = ['RGI60-17.14602', 'RGI60-17.15077', 'RGI60-17.15112', 
               'RGI60-17.15452', 'RGI60-17.15496', 'RGI60-17.15636']

print("\n" + "="*70)
print("INVESTIGATING PROBLEM GLACIERS")
print("="*70)

for rgi_id in problem_ids:
    print(f"\n{rgi_id}:")
    
    # Check if glacier exists in each dataset
    cr2_exists = rgi_id in [str(x) for x in ds_cr2['rgi_id'].values]
    cru_exists = rgi_id in [str(x) for x in ds_cru['rgi_id'].values]
    era5_exists = rgi_id in [str(x) for x in ds_era5['rgi_id'].values]
    
    print(f"  In CR2MET: {cr2_exists}")
    print(f"  In CRU:    {cru_exists}")
    print(f"  In ERA5:   {era5_exists}")
    
    if cr2_exists:
        area_1999 = float(ds_cr2['area'].sel(time=1999, rgi_id=rgi_id).values) / 1e6
        print(f"  CR2MET area (1999): {area_1999:.4f} km²")
    
    if cru_exists:
        area_1999 = float(ds_cru['area'].sel(time=1999, rgi_id=rgi_id).values) / 1e6
        print(f"  CRU area (1999):    {area_1999:.4f} km²")
    
    if era5_exists:
        area_1999 = float(ds_era5['area'].sel(time=1999, rgi_id=rgi_id).values) / 1e6
        print(f"  ERA5 area (1999):   {area_1999:.4f} km²")

print("\n" + "="*70)

ds_cr2.close()
ds_cru.close()
ds_era5.close()


INVESTIGATING PROBLEM GLACIERS

RGI60-17.14602:
  In CR2MET: True
  In CRU:    True
  In ERA5:   True
  CR2MET area (1999): 0.7680 km²
  CRU area (1999):    0.7750 km²
  ERA5 area (1999):   0.7833 km²

RGI60-17.15077:
  In CR2MET: True
  In CRU:    True
  In ERA5:   True
  CR2MET area (1999): 0.0320 km²
  CRU area (1999):    0.0000 km²
  ERA5 area (1999):   0.0320 km²

RGI60-17.15112:
  In CR2MET: True
  In CRU:    True
  In ERA5:   True
  CR2MET area (1999): 0.0400 km²
  CRU area (1999):    0.0000 km²
  ERA5 area (1999):   0.0400 km²

RGI60-17.15452:
  In CR2MET: True
  In CRU:    True
  In ERA5:   True
  CR2MET area (1999): 0.0490 km²
  CRU area (1999):    0.0000 km²
  ERA5 area (1999):   0.0490 km²

RGI60-17.15496:
  In CR2MET: True
  In CRU:    True
  In ERA5:   True
  CR2MET area (1999): 0.0280 km²
  CRU area (1999):    nan km²
  ERA5 area (1999):   0.0280 km²

RGI60-17.15636:
  In CR2MET: True
  In CRU:    True
  In ERA5:   True
  CR2MET area (1999): 1.0317 km²
  CRU area (1999)

In [16]:
import xarray as xr
import numpy as np
import pandas as pd

ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')
ds_cru = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CRU/DA1/run_output_2000_2019_hydro_CRU_DA1.nc')
ds_era5 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/ERA5/DA1/run_output_2000_2019_hydro_ERA5_DA1.nc')

# Find glaciers that succeeded in ALL datasets
glacier_ids = [str(x) for x in ds_cr2['rgi_id'].values]

valid_glaciers = []
for rgi_id in glacier_ids:
    # Check year 1999 area > 0 and not NaN in all datasets
    cr2_area = float(ds_cr2['area'].sel(time=1999, rgi_id=rgi_id).values)
    cru_area = float(ds_cru['area'].sel(time=1999, rgi_id=rgi_id).values)
    era5_area = float(ds_era5['area'].sel(time=1999, rgi_id=rgi_id).values)
    
    if (cr2_area > 0 and cru_area > 0 and era5_area > 0 and 
        not np.isnan(cr2_area) and not np.isnan(cru_area) and not np.isnan(era5_area)):
        valid_glaciers.append(rgi_id)

print(f"\n{'='*70}")
print("VALID GLACIERS (succeeded in ALL datasets)")
print(f"{'='*70}")
print(f"Total glaciers in NetCDF: {len(glacier_ids)}")
print(f"Valid glaciers (all datasets): {len(valid_glaciers)}")
print(f"Excluded glaciers: {len(glacier_ids) - len(valid_glaciers)}")

# Calculate SMB using ONLY valid glaciers
def calculate_smb_valid_only(ds, valid_ids):
    vol_2000 = 0
    vol_2020 = 0
    area_2000 = 0
    
    for rgi_id in valid_ids:
        vol_2000 += float(ds['volume'].sel(time=2000, rgi_id=rgi_id).values)
        vol_2020 += float(ds['volume'].sel(time=2020, rgi_id=rgi_id).values)
        area_2000 += float(ds['area'].sel(time=2000, rgi_id=rgi_id).values)
    
    return ((vol_2020 - vol_2000) / area_2000 / 20) * 1000  # mm/yr

smb_cr2 = calculate_smb_valid_only(ds_cr2, valid_glaciers)
smb_cru = calculate_smb_valid_only(ds_cru, valid_glaciers)
smb_era5 = calculate_smb_valid_only(ds_era5, valid_glaciers)

# GMB for same glaciers
from oggm import utils
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

gmb_valid = np.average(
    geodetic_ref.loc[valid_glaciers, 'dmdtda'] * 1000,
    weights=geodetic_ref.loc[valid_glaciers, 'area']
)
gmb_error = np.average(
    geodetic_ref.loc[valid_glaciers, 'err_dmdtda'] * 1000,
    weights=geodetic_ref.loc[valid_glaciers, 'area']
)

print(f"\n{'='*70}")
print("CORRECTED COMPARISON (only valid glaciers)")
print(f"{'='*70}")
print(f"\nCR2MET:")
print(f"  SMB: {smb_cr2:.1f} mm/yr")
print(f"  GMB: {gmb_valid:.1f} ± {gmb_error:.1f} mm/yr")
print(f"  Bias: {smb_cr2 - gmb_valid:+.1f} mm/yr")

print(f"\nCRU:")
print(f"  SMB: {smb_cru:.1f} mm/yr")
print(f"  GMB: {gmb_valid:.1f} ± {gmb_error:.1f} mm/yr")
print(f"  Bias: {smb_cru - gmb_valid:+.1f} mm/yr")

print(f"\nERA5:")
print(f"  SMB: {smb_era5:.1f} mm/yr")
print(f"  GMB: {gmb_valid:.1f} ± {gmb_error:.1f} mm/yr")
print(f"  Bias: {smb_era5 - gmb_valid:+.1f} mm/yr")

print(f"{'='*70}\n")

ds_cr2.close()
ds_cru.close()
ds_era5.close()


VALID GLACIERS (succeeded in ALL datasets)
Total glaciers in NetCDF: 422
Valid glaciers (all datasets): 417
Excluded glaciers: 5

CORRECTED COMPARISON (only valid glaciers)

CR2MET:
  SMB: -94.3 mm/yr
  GMB: -98.9 ± 152.5 mm/yr
  Bias: +4.5 mm/yr

CRU:
  SMB: -62.2 mm/yr
  GMB: -98.9 ± 152.5 mm/yr
  Bias: +36.7 mm/yr

ERA5:
  SMB: -108.2 mm/yr
  GMB: -98.9 ± 152.5 mm/yr
  Bias: -9.4 mm/yr



In [18]:
# What period does GMB actually cover?
from oggm import utils
geodetic_ref = utils.get_geodetic_mb_dataframe()

# Check all available periods
print("Available GMB periods:")
print(geodetic_ref['period'].unique())

# Check what we're using
gmb_2000_2020 = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']
print(f"\nUsing period: 2000-01-01_2020-01-01")
print(f"Number of glaciers: {len(gmb_2000_2020)}")

# But if NetCDF year labels are off by 1...
# Year 1999 in NetCDF = Real year 2000
# Year 2020 in NetCDF = Real year 2019?
# Then we're comparing SMB(1999-2020 NetCDF) vs GMB(2000-2020 real)

Available GMB periods:
['2000-01-01_2010-01-01' '2000-01-01_2020-01-01' '2010-01-01_2020-01-01']

Using period: 2000-01-01_2020-01-01
Number of glaciers: 215547


In [19]:
import xarray as xr

ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')

print("\n" + "="*70)
print("UNDERSTANDING THE TIME DIMENSION")
print("="*70)

print(f"\nNetCDF time values: {ds_cr2['time'].values}")

# We know:
# - NetCDF "year 1999" = RGI inventory (~real year 2000)
# - Simulation started at ys=1999

# So the mapping might be:
print("\nHypothesis 1: NetCDF years = Real years - 1")
print("  NetCDF 1999 → Real 2000 (RGI inventory)")
print("  NetCDF 2000 → Real 2001")
print("  NetCDF 2020 → Real 2021")
print("  → We should use: (Vol[2019] - Vol[1999]) / Area[1999] / 20")

print("\nHypothesis 2: NetCDF years = Simulation years starting from 1999")
print("  NetCDF 1999 → Initial state (RGI ~2000)")
print("  NetCDF 2000 → After 1 year (Real 2000)")
print("  NetCDF 2020 → After 21 years (Real 2020)")
print("  → We should use: (Vol[2020] - Vol[2000]) / Area[2000] / 20")
print("  → Or: (Vol[2020] - Vol[1999]) / Area[1999] / 21")

# What does the calibration code say?
print("\n" + "="*70)
print("CHECKING CALIBRATION")
print("="*70)
print("\nFrom 01_ script:")
print("  workflow.execute_entity_task(tasks.mu_star_calibration_from_geodetic_mb,")
print("                                gdirs, ref_period='2000-01-01_2020-01-01')")
print("\n  ys=1999  (simulation starts at year 1999)")

print("\nSo OGGM calibrates to match:")
print("  GMB: 2000-01-01 to 2020-01-01 (real calendar dates)")
print("  SMB: Year 1999 to Year ???? in NetCDF")

print("\nWe need to figure out: What NetCDF year = Real Jan 2020?")

ds_cr2.close()


UNDERSTANDING THE TIME DIMENSION

NetCDF time values: [1999. 2000. 2001. 2002. 2003. 2004. 2005. 2006. 2007. 2008. 2009. 2010.
 2011. 2012. 2013. 2014. 2015. 2016. 2017. 2018. 2019. 2020. 2021. 2022.]

Hypothesis 1: NetCDF years = Real years - 1
  NetCDF 1999 → Real 2000 (RGI inventory)
  NetCDF 2000 → Real 2001
  NetCDF 2020 → Real 2021
  → We should use: (Vol[2019] - Vol[1999]) / Area[1999] / 20

Hypothesis 2: NetCDF years = Simulation years starting from 1999
  NetCDF 1999 → Initial state (RGI ~2000)
  NetCDF 2000 → After 1 year (Real 2000)
  NetCDF 2020 → After 21 years (Real 2020)
  → We should use: (Vol[2020] - Vol[2000]) / Area[2000] / 20
  → Or: (Vol[2020] - Vol[1999]) / Area[1999] / 21

CHECKING CALIBRATION

From 01_ script:
  workflow.execute_entity_task(tasks.mu_star_calibration_from_geodetic_mb,
                                gdirs, ref_period='2000-01-01_2020-01-01')

  ys=1999  (simulation starts at year 1999)

So OGGM calibrates to match:
  GMB: 2000-01-01 to 2020-01-0

In [20]:
import xarray as xr
import numpy as np
from oggm import utils

ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')

# Load valid glaciers (the 417 that succeeded in all datasets)
# From earlier, we know these are the ones with area > 0 in year 1999 for all datasets

# Load GMB
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

# Get valid glacier IDs (simplified - assuming we have them from before)
glacier_ids = [str(x) for x in ds_cr2['rgi_id'].values]

# For now, test with ALL glaciers to see the pattern
print("\n" + "="*70)
print("TESTING DIFFERENT YEAR RANGES")
print("="*70)

# Calculate GMB for all DA1 glaciers
da1_glaciers = [g for g in glacier_ids if g in geodetic_ref.index]
gmb_value = np.average(
    geodetic_ref.loc[da1_glaciers, 'dmdtda'] * 1000,
    weights=geodetic_ref.loc[da1_glaciers, 'area']
)

print(f"\nGMB (2000-01-01 to 2020-01-01): {gmb_value:.1f} mm/yr")

# Test different hypotheses
tests = [
    # (start_year, end_year, n_years, description)
    (1999, 2019, 20, "Hypothesis 1: NetCDF 1999-2019 = Real 2000-2020"),
    (2000, 2020, 20, "Current method: NetCDF 2000-2020"),
    (1999, 2020, 21, "Hypothesis 2: NetCDF 1999-2020 (21 years)"),
]

print(f"\n{'Method':<50} {'SMB':<12} {'Bias':<12}")
print("-" * 74)

for start, end, n_years, desc in tests:
    # Calculate SMB
    vol_start = ds_cr2['volume'].sel(time=start).sum()
    vol_end = ds_cr2['volume'].sel(time=end).sum()
    
    # Which area to use?
    # Try with start year area (most common approach)
    area_start = ds_cr2['area'].sel(time=start).sum()
    
    smb = float((vol_end - vol_start) / area_start / n_years * 1000)
    bias = smb - gmb_value
    
    print(f"{desc:<50} {smb:<12.1f} {bias:<12.1f}")

print("\n" + "="*70)
print("Which hypothesis gives bias closest to 0?")
print("="*70)

ds_cr2.close()


TESTING DIFFERENT YEAR RANGES

GMB (2000-01-01 to 2020-01-01): -98.7 mm/yr

Method                                             SMB          Bias        
--------------------------------------------------------------------------
Hypothesis 1: NetCDF 1999-2019 = Real 2000-2020    -65.1        33.6        
Current method: NetCDF 2000-2020                   -93.8        4.9         
Hypothesis 2: NetCDF 1999-2020 (21 years)          -82.1        16.6        

Which hypothesis gives bias closest to 0?


In [21]:
import xarray as xr
import numpy as np
from oggm import utils

ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')

# Load GMB
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

glacier_ids = [str(x) for x in ds_cr2['rgi_id'].values]
da1_glaciers = [g for g in glacier_ids if g in geodetic_ref.index]

gmb_value = np.average(
    geodetic_ref.loc[da1_glaciers, 'dmdtda'] * 1000,
    weights=geodetic_ref.loc[da1_glaciers, 'area']
)

print("\n" + "="*70)
print("COMPARING TWO SMB CALCULATION METHODS")
print("="*70)

# Method 1: Total volume change (what we've been doing)
vol_2000 = ds_cr2['volume'].sel(time=2000).sum()
vol_2020 = ds_cr2['volume'].sel(time=2020).sum()
area_2000 = ds_cr2['area'].sel(time=2000).sum()

smb_method1 = float((vol_2020 - vol_2000) / area_2000 / 20 * 1000)

print(f"\nMethod 1: (Vol₂₀₂₀ - Vol₂₀₀₀) / Area₂₀₀₀ / 20 years")
print(f"  SMB: {smb_method1:.1f} mm/yr")
print(f"  GMB: {gmb_value:.1f} mm/yr")
print(f"  Bias: {smb_method1 - gmb_value:+.1f} mm/yr")

# Method 2: Average of annual mass balance rates
# Calculate year-to-year changes
annual_smb = []
for year in range(2000, 2020):
    vol_start = ds_cr2['volume'].sel(time=year).sum()
    vol_end = ds_cr2['volume'].sel(time=year+1).sum()
    area_year = ds_cr2['area'].sel(time=year).sum()
    
    mb_year = float((vol_end - vol_start) / area_year * 1000)
    annual_smb.append(mb_year)

smb_method2 = np.mean(annual_smb)

print(f"\nMethod 2: Average of annual SMB (2000→2001, 2001→2002, ..., 2019→2020)")
print(f"  SMB: {smb_method2:.1f} mm/yr")
print(f"  GMB: {gmb_value:.1f} mm/yr")
print(f"  Bias: {smb_method2 - gmb_value:+.1f} mm/yr")

print(f"\n" + "="*70)
print("Does averaging annual rates give bias ≈ 0?")
print("="*70)

ds_cr2.close()


COMPARING TWO SMB CALCULATION METHODS

Method 1: (Vol₂₀₂₀ - Vol₂₀₀₀) / Area₂₀₀₀ / 20 years
  SMB: -93.8 mm/yr
  GMB: -98.7 mm/yr
  Bias: +4.9 mm/yr

Method 2: Average of annual SMB (2000→2001, 2001→2002, ..., 2019→2020)
  SMB: -95.3 mm/yr
  GMB: -98.7 mm/yr
  Bias: +3.4 mm/yr

Does averaging annual rates give bias ≈ 0?


In [22]:
import xarray as xr
import numpy as np
from oggm import utils

ds_cr2 = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')

# Load GMB
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

glacier_ids = [str(x) for x in ds_cr2['rgi_id'].values]
da1_glaciers = [g for g in glacier_ids if g in geodetic_ref.index]

gmb_value = np.average(
    geodetic_ref.loc[da1_glaciers, 'dmdtda'] * 1000,
    weights=geodetic_ref.loc[da1_glaciers, 'area']
)

print("\n" + "="*70)
print("TESTING WITH REFERENCE AREA (from run settings)")
print("="*70)

# Look at the run settings - remember this line?
# ref_area_from_y0 = True
# This means: "keep the reference area as the year 0 of the simulation"

# Year 0 of simulation = 1999 in NetCDF
area_ref = ds_cr2['area'].sel(time=1999).sum()

print(f"\nReminder from 01_ script:")
print(f"  ref_area_from_y0 = True")
print(f"  → Reference area = year 1999 (simulation start)")

# Test using year 1999 as reference
print(f"\n{'Method':<60} {'SMB':<12} {'Bias':<12}")
print("-" * 84)

# Method A: 1999-2020, ref area = 1999
vol_1999 = ds_cr2['volume'].sel(time=1999).sum()
vol_2020 = ds_cr2['volume'].sel(time=2020).sum()
smb_a = float((vol_2020 - vol_1999) / area_ref / 21 * 1000)
print(f"{'(Vol₂₀₂₀ - Vol₁₉₉₉) / Area₁₉₉₉ / 21 years':<60} {smb_a:<12.1f} {smb_a - gmb_value:<12.1f}")

# Method B: 1999-2019, ref area = 1999  
vol_2019 = ds_cr2['volume'].sel(time=2019).sum()
smb_b = float((vol_2019 - vol_1999) / area_ref / 20 * 1000)
print(f"{'(Vol₂₀₁₉ - Vol₁₉₉₉) / Area₁₉₉₉ / 20 years':<60} {smb_b:<12.1f} {smb_b - gmb_value:<12.1f}")

# Method C: 2000-2020, ref area = 1999
vol_2000 = ds_cr2['volume'].sel(time=2000).sum()
smb_c = float((vol_2020 - vol_2000) / area_ref / 20 * 1000)
print(f"{'(Vol₂₀₂₀ - Vol₂₀₀₀) / Area₁₉₉₉ / 20 years (different ref area!)':<60} {smb_c:<12.1f} {smb_c - gmb_value:<12.1f}")

print("\n" + "="*70)

# Also check: What does the calibration code actually compute?
print("\nWhat does mu_star_calibration actually calculate?")
print("Looking at the code, it should compute the same thing...")
print("\nMaybe the +3-5 mm/yr bias IS expected and acceptable?")
print("Let me check what Ale's old results show...")

ds_cr2.close()


TESTING WITH REFERENCE AREA (from run settings)

Reminder from 01_ script:
  ref_area_from_y0 = True
  → Reference area = year 1999 (simulation start)

Method                                                       SMB          Bias        
------------------------------------------------------------------------------------
(Vol₂₀₂₀ - Vol₁₉₉₉) / Area₁₉₉₉ / 21 years                    -82.1        16.6        
(Vol₂₀₁₉ - Vol₁₉₉₉) / Area₁₉₉₉ / 20 years                    -65.1        33.6        
(Vol₂₀₂₀ - Vol₂₀₀₀) / Area₁₉₉₉ / 20 years (different ref area!) -96.3        2.4         


What does mu_star_calibration actually calculate?
Looking at the code, it should compute the same thing...

Maybe the +3-5 mm/yr bias IS expected and acceptable?
Let me check what Ale's old results show...


In [23]:
import xarray as xr
import numpy as np
import pandas as pd
from oggm import utils
import os

print("\n" + "="*70)
print("COMPARING YOUR RESULTS TO ALVARO'S RESULTS")
print("="*70)

# Check what files Alvaro has
alvaro_path = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/alvaros_already_run/DA1/'
print(f"\nAlvaro's DA1 files:")
files = os.listdir(alvaro_path)
for f in sorted(files):
    print(f"  {f}")

# Look for his NetCDF output
alvaro_nc_files = [f for f in files if f.endswith('.nc')]
print(f"\nNetCDF files: {alvaro_nc_files}")

# Also check for comparison CSV
comparison_files = [f for f in files if 'comparacion' in f.lower() or 'gmb' in f.lower()]
print(f"Comparison files: {comparison_files}")

# If he has a comparison file, read it
if comparison_files:
    for comp_file in comparison_files:
        print(f"\n{'='*70}")
        print(f"Reading: {comp_file}")
        print(f"{'='*70}")
        
        comp_path = os.path.join(alvaro_path, comp_file)
        df = pd.read_csv(comp_path, header=None)
        print(df.to_string())

# If he has NetCDF, calculate his SMB
if alvaro_nc_files:
    print(f"\n{'='*70}")
    print("Calculating Alvaro's SMB from NetCDF")
    print(f"{'='*70}")
    
    alvaro_nc = xr.open_dataset(os.path.join(alvaro_path, alvaro_nc_files[0]))
    
    # Calculate same way we did
    vol_2000_alv = alvaro_nc['volume'].sel(time=2000).sum()
    vol_2020_alv = alvaro_nc['volume'].sel(time=2020).sum()
    area_2000_alv = alvaro_nc['area'].sel(time=2000).sum()
    
    smb_alvaro = float((vol_2020_alv - vol_2000_alv) / area_2000_alv / 20 * 1000)
    
    print(f"  Alvaro's SMB (2000-2020): {smb_alvaro:.1f} mm/yr")
    print(f"  Your SMB (2000-2020):     -93.8 mm/yr")
    print(f"  Difference: {abs(smb_alvaro + 93.8):.1f} mm/yr")
    
    alvaro_nc.close()

print("\n" + "="*70)


COMPARING YOUR RESULTS TO ALVARO'S RESULTS

Alvaro's DA1 files:
  RGI_DA1
  per_glacier
  run_output_2000_2019_hydro_TC_DA1.nc

NetCDF files: ['run_output_2000_2019_hydro_TC_DA1.nc']
Comparison files: []

Calculating Alvaro's SMB from NetCDF
  Alvaro's SMB (2000-2020): -103.2 mm/yr
  Your SMB (2000-2020):     -93.8 mm/yr
  Difference: 9.4 mm/yr



In [24]:
import xarray as xr
import numpy as np
from oggm import utils

# Your results
your_smb = -93.8
gmb_value = -98.7

# Alvaro's results
alvaro_smb = -103.2

print("\n" + "="*70)
print("BIAS COMPARISON")
print("="*70)

print(f"\nGMB (Hugonnet): {gmb_value:.1f} mm/yr")
print(f"\nYour results:")
print(f"  SMB: {your_smb:.1f} mm/yr")
print(f"  Bias: {your_smb - gmb_value:+.1f} mm/yr ({abs(your_smb - gmb_value)/abs(gmb_value)*100:.1f}%)")

print(f"\nAlvaro's results:")
print(f"  SMB: {alvaro_smb:.1f} mm/yr")
print(f"  Bias: {alvaro_smb - gmb_value:+.1f} mm/yr ({abs(alvaro_smb - gmb_value)/abs(gmb_value)*100:.1f}%)")

print(f"\n{'='*70}")
print("KEY FINDING:")
print(f"{'='*70}")
print(f"Your bias:    {your_smb - gmb_value:+.1f} mm/yr")
print(f"Alvaro's bias: {alvaro_smb - gmb_value:+.1f} mm/yr")
print(f"\nBoth have biases of 4-5 mm/yr magnitude!")
print(f"This suggests the bias IS expected and normal.")

# Check if Alvaro used same number of glaciers
alvaro_nc = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/alvaros_already_run/DA1/run_output_2000_2019_hydro_TC_DA1.nc')
your_nc = xr.open_dataset('/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc')

print(f"\n{'='*70}")
print("GLACIER COUNTS")
print(f"{'='*70}")
print(f"Your run:    {len(your_nc['rgi_id'])} glaciers")
print(f"Alvaro's run: {len(alvaro_nc['rgi_id'])} glaciers")

# Check if same glaciers
your_ids = set([str(x) for x in your_nc['rgi_id'].values])
alvaro_ids = set([str(x) for x in alvaro_nc['rgi_id'].values])

if your_ids == alvaro_ids:
    print("✅ Same glacier set!")
else:
    print(f"❌ Different glaciers!")
    print(f"  Only in yours: {len(your_ids - alvaro_ids)}")
    print(f"  Only in Alvaro's: {len(alvaro_ids - your_ids)}")

alvaro_nc.close()
your_nc.close()

print(f"\n{'='*70}")
print("CONCLUSION")
print(f"{'='*70}")
print("Both you and Alvaro get ~4-5 mm/yr biases.")
print("This appears to be NORMAL and EXPECTED behavior!")
print("The calibration minimizes bias but doesn't force it to exactly zero.")
print(f"{'='*70}\n")


BIAS COMPARISON

GMB (Hugonnet): -98.7 mm/yr

Your results:
  SMB: -93.8 mm/yr
  Bias: +4.9 mm/yr (5.0%)

Alvaro's results:
  SMB: -103.2 mm/yr
  Bias: -4.5 mm/yr (4.6%)

KEY FINDING:
Your bias:    +4.9 mm/yr
Alvaro's bias: -4.5 mm/yr

Both have biases of 4-5 mm/yr magnitude!
This suggests the bias IS expected and normal.

GLACIER COUNTS
Your run:    422 glaciers
Alvaro's run: 422 glaciers
✅ Same glacier set!

CONCLUSION
Both you and Alvaro get ~4-5 mm/yr biases.
This appears to be NORMAL and EXPECTED behavior!
The calibration minimizes bias but doesn't force it to exactly zero.



In [25]:
import xarray as xr
import numpy as np
import os

print("\n" + "="*70)
print("ANALYZING ALL 10 CLUSTERS - VALID GLACIER COUNTS")
print("="*70)

clusters = ['OT3', 'DA1', 'DA2', 'DA3', 'WA1', 'WA2', 'WA3', 'WA4', 'WA5', 'WA6']
datasets = ['CR2MET', 'CRU', 'ERA5']

base_path = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/'

all_cluster_stats = []

for cluster in clusters:
    print(f"\n{cluster}:")
    
    # Load all three NetCDF files for this cluster
    nc_files = {}
    for dataset in datasets:
        if dataset == 'CR2MET':
            nc_path = f'{base_path}CR2MET/{cluster}/run_output_2000_2019_hydro_TC_{cluster}.nc'
        else:
            nc_path = f'{base_path}{dataset}/{cluster}/run_output_2000_2019_hydro_{dataset}_{cluster}.nc'
        
        if os.path.exists(nc_path):
            nc_files[dataset] = xr.open_dataset(nc_path)
        else:
            print(f"  ✗ {dataset} file not found")
    
    if len(nc_files) != 3:
        print(f"  ⚠ Missing datasets, skipping cluster")
        continue
    
    # Get glacier IDs for each dataset
    glacier_ids = {}
    for dataset, ds in nc_files.items():
        glacier_ids[dataset] = [str(x) for x in ds['rgi_id'].values]
    
    # Find glaciers valid in ALL datasets (area > 0 and not NaN at year 1999)
    valid_glaciers = []
    
    for rgi_id in glacier_ids['CR2MET']:
        # Check if glacier exists and has valid area in all datasets
        valid = True
        
        for dataset, ds in nc_files.items():
            if rgi_id not in glacier_ids[dataset]:
                valid = False
                break
            
            try:
                area_1999 = float(ds['area'].sel(time=1999, rgi_id=rgi_id).values)
                if area_1999 <= 0 or np.isnan(area_1999):
                    valid = False
                    break
            except:
                valid = False
                break
        
        if valid:
            valid_glaciers.append(rgi_id)
    
    # Stats
    total = len(glacier_ids['CR2MET'])
    valid = len(valid_glaciers)
    failed = total - valid
    
    print(f"  Total glaciers (RGI): {total}")
    print(f"  Valid in all 3 datasets: {valid}")
    print(f"  Failed in at least 1 dataset: {failed}")
    
    all_cluster_stats.append({
        'cluster': cluster,
        'total': total,
        'valid': valid,
        'failed': failed,
        'cr2met_count': len(glacier_ids.get('CR2MET', [])),
        'cru_count': len(glacier_ids.get('CRU', [])),
        'era5_count': len(glacier_ids.get('ERA5', [])),
    })
    
    # Close files
    for ds in nc_files.values():
        ds.close()

# Summary
import pandas as pd
df_stats = pd.DataFrame(all_cluster_stats)

print(f"\n{'='*70}")
print("SUMMARY - ALL CLUSTERS")
print(f"{'='*70}")
print(df_stats.to_string(index=False))

print(f"\n{'='*70}")
print("TOTALS ACROSS ALL CHILE")
print(f"{'='*70}")
print(f"Total glaciers (all clusters): {df_stats['total'].sum()}")
print(f"Valid in all 3 datasets: {df_stats['valid'].sum()}")
print(f"Failed in at least 1: {df_stats['failed'].sum()}")
print(f"Success rate: {df_stats['valid'].sum() / df_stats['total'].sum() * 100:.1f}%")

print(f"\nGlacier counts by dataset:")
print(f"  CR2MET: {df_stats['cr2met_count'].sum()}")
print(f"  CRU:    {df_stats['cru_count'].sum()}")
print(f"  ERA5:   {df_stats['era5_count'].sum()}")

print(f"{'='*70}\n")


ANALYZING ALL 10 CLUSTERS - VALID GLACIER COUNTS

OT3:
  Total glaciers (RGI): 34
  Valid in all 3 datasets: 34
  Failed in at least 1 dataset: 0

DA1:
  Total glaciers (RGI): 422
  Valid in all 3 datasets: 417
  Failed in at least 1 dataset: 5

DA2:
  Total glaciers (RGI): 200
  Valid in all 3 datasets: 200
  Failed in at least 1 dataset: 0

DA3:
  Total glaciers (RGI): 410
  Valid in all 3 datasets: 410
  Failed in at least 1 dataset: 0

WA1:
  Total glaciers (RGI): 678
  Valid in all 3 datasets: 670
  Failed in at least 1 dataset: 8

WA2:
  Total glaciers (RGI): 2924
  Valid in all 3 datasets: 2911
  Failed in at least 1 dataset: 13

WA3:
  Total glaciers (RGI): 447
  Valid in all 3 datasets: 445
  Failed in at least 1 dataset: 2

WA4:
  Total glaciers (RGI): 4749
  Valid in all 3 datasets: 4701
  Failed in at least 1 dataset: 48

WA5:
  Total glaciers (RGI): 711
  Valid in all 3 datasets: 704
  Failed in at least 1 dataset: 7

WA6:
  Total glaciers (RGI): 2671
  Valid in all 3 dat